# ML-8A — Temporal Boosting ve Karar Katmanları

Bu notebook yalnız raporlama/görselleştirme içindir. Hesap `scripts/run_ml8a_temporal_boosting.py` içinde yaşar. Blind holdout açılmamıştır.

In [ ]:
import json
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sead_payload = json.loads((ROOT / 'artifacts/ml8a/uav_sead/full_matrix_gapfix/metrics.json').read_text())
alfa_payload = json.loads((ROOT / 'artifacts/ml8a/alfa/full_matrix_gapfix/metrics.json').read_text())
sead = pd.DataFrame(sead_payload['results']).assign(source='uav_sead')
alfa = pd.DataFrame(alfa_payload['results']).assign(source='alfa')
results = pd.concat([sead, alfa], ignore_index=True)
results.groupby('source').size()

## Skor kalitesi (5 seed ortalaması)

In [ ]:
window = results.groupby(['source','score_source'])[['window_auroc','window_auprc','prevalence']].mean().round(3)
window

## Operasyonel matris

In [ ]:
operational = results.groupby(['source','score_source','decision','budget'])[['event_onset_recall','false_alarms_per_hour','avg_detection_time_s','max_detection_time_s']].agg(['mean','std']).round(3)
operational

## Feature importance

In [ ]:
importance = {}
for source in ['uav_sead', 'alfa']:
    path = ROOT / f'artifacts/ml8a/{source}/full_matrix_gapfix/split_00/feature_importance.json'
    importance[source] = pd.DataFrame(json.loads(path.read_text())).head(20)
importance

## Aile / fault kırılımı

In [ ]:
family_rows = []
for _, row in results.iterrows():
    for family, values in row['family_onset_recall'].items():
        family_rows.append({'source': row.source, 'score_source': row.score_source, 'decision': row.decision, 'budget': row.budget, 'family': family, **values})
families = pd.DataFrame(family_rows)
families.groupby(['source','score_source','decision','budget','family'])[['detected','events']].sum().assign(recall=lambda x: x.detected/x.events)

## Gate değerlendirmesi

- Gate A: **GEÇTİ** — causal descriptor, split/holdout, train-only scaler ve hash kontrolleri tamam.
- Gate B: **KALDI** — SEAD LightGBM AUPRC 0.349 < IF 0.385; ALFA LightGBM 0.843 < IF 0.858 < LSTM 0.872.
- Gate C: matris düzeyinde **GEÇTİ** — ALFA IF+CUSUM advisory 0.625 recall / 7.91 FA-saat; yeni LightGBM için ve SEAD'de **KALDI**.
- Mevcut SEAD LSTM baseline: **N/A**. `lstm_retrained` ayrı ve geriye dönük baseline değildir.
- ALFA, fault/aile kırılımları ve resmi Avg/Max Detection Time tamamlandı; tuning veya holdout açılması yapılmadı. Sonraki faz ML-8C'dir.